#### Imports needed

In [2]:
import subprocess
subprocess.run(["pip", "install", "azure-cognitiveservices-vision-customvision", "msrest", "--quiet"])
import importlib, site
importlib.invalidate_caches()
importlib.reload(site)
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
from msrest.authentication import ApiKeyCredentials
from datetime import datetime
from pyspark.sql import Row
import os

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 4, Finished, Available, Finished, False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.


#### Parameter cell

In [1]:
# Tagged Parameters cell
cropped_path = "Files/development/cropped/elsie_mas_2_people_20260602_181602_0.jpg"  # default for testing

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 3, Finished, Available, Finished, False)

#### Credentials

In [3]:
vault_url  = "https://cv-training-key.vault.azure.net/"

# Prediction — for running inference
key_p      = notebookutils.credentials.getSecret(vault_url, "cv-prediction-api-key")
endpoint_p = notebookutils.credentials.getSecret(vault_url, "cv-endpoint-p-key")

# Training — READ ONLY, used only to discover available projects dynamically
key_t      = notebookutils.credentials.getSecret(vault_url, "cv-training-api-key")
endpoint_t = notebookutils.credentials.getSecret(vault_url, "cv-endpoint-t-key")

predictor = CustomVisionPredictionClient(
    endpoint_p,
    ApiKeyCredentials(in_headers={"Prediction-key": key_p})
)
trainer = CustomVisionTrainingClient(
    endpoint_t,
    ApiKeyCredentials(in_headers={"Training-key": key_t})
)

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 5, Finished, Available, Finished, False)

#### Get absolute path cell

In [4]:
# cropped_path is already a full abfss:// path when injected by @item()
# For manual runs it's a relative path — handle both cases
if cropped_path.startswith("abfss://"):
    abs_cropped = cropped_path
else:
    files_listing  = notebookutils.fs.ls("Files")
    lakehouse_root = files_listing[0].path.split("/Files/")[0]
    abs_cropped    = f"{lakehouse_root}/{cropped_path}"

local_tmp = "/tmp/inference_input.jpg"
notebookutils.fs.cp(abs_cropped, f"file:{local_tmp}")

filename = os.path.basename(cropped_path)
print(f"✅ Image ready for inference: {filename}")

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 6, Finished, Available, Finished, False)

✅ Image ready for inference: elsie_mas_2_people_20260602_181602_0.jpg


#### Run inference against all models cell

In [5]:
rows = []
timestamp = datetime.now()

# Note: trainer is used READ-ONLY to dynamically discover all published pet-classifier projects at runtime.
# This avoids hardcoding model sizes and scales automatically when new sizes are added to pl_ml_training.
all_projects = {
    p.name: p for p in trainer.get_projects()
    if p.name.startswith("pet-classifier-")
}

print(f"✅ Discovered {len(all_projects)} models: {list(all_projects.keys())}")

for project_name, project in all_projects.items():

    # Extract size from project name e.g. "pet-classifier-128" -> "128"
    size = project_name.replace("pet-classifier-","")
    publish_name = f"publish_{size}"
    

    try:
        with open(local_tmp, "rb") as img:
            results = predictor.classify_image(
                project.id,
                publish_name,
                img.read()
            )

        if not results.predictions:
            print(f"⚠️ No predictions for size {size}")
            continue

        top = results.predictions[0]
        predicted = top.tag_name
        confidence = float(top.probability)

        rows.append(Row(
            image_name = filename,
            model_size = size,
            predicted = predicted,
            confidence = confidence,
            cropped_image_url = f"https://onelake.dfs.fabric.microsoft.com/lkh_pets.Lakehouse/Files/development/cropped/{filename}",
            timestamp = timestamp
        ))

        print(f"✅ Model {size}: '{predicted}' ({confidence:.2%})")

    except Exception as e:
        print(f"❌ Error on model {size}: {e}")

print(f"\n📦 Inference complete: {len(rows)} model(s) evaluated")

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 7, Finished, Available, Finished, False)

✅ Discovered 2 models: ['pet-classifier-224', 'pet-classifier-128']
✅ Model 224: 'gato_phil' (32.83%)
✅ Model 128: 'gato_phil' (67.65%)

📦 Inference complete: 2 model(s) evaluated


#### Save to inference_metrics cell

In [6]:
if rows:
    inference_df = spark.createDataFrame(rows)
    inference_df.write.mode("append").saveAsTable("inference_metrics")
    print(f"✅ Saved {len(rows)} inference results for '{filename}'")
else:
    print("⚠️ No results to save")

StatementMeta(, e3f22906-a0e8-48cc-a5b9-93aa36ebe023, 8, Finished, Available, Finished, False)

✅ Saved 2 inference results for 'elsie_mas_2_people_20260602_181602_0.jpg'
